In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l1.3 Softmax

Softmax turns raw scores (logits) into a probability distribution: exponentiate,
then normalize. Build it by hand, then check it against PyTorch.

In [ ]:
from torch.nn import functional as F

logits = torch.tensor([2.0, 1.0, 0.1])
by_hand = logits.exp() / logits.exp().sum()
builtin = F.softmax(logits, dim=-1)
print("by hand: ", by_hand)
print("F.softmax:", builtin)

Both agree, and the result sums to 1: it is a valid distribution over the
next token.

In [ ]:
assert torch.allclose(by_hand, builtin)
assert abs(builtin.sum().item() - 1.0) < 1e-6